In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.stats import spearmanr
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import math
import random

torch.manual_seed(0); np.random.seed(0)
train = datasets.MNIST('data', train=True, download=True)
test = datasets.MNIST('data', train=False, download=True)

In [3]:
class RotatedMNIST(Dataset):
    def __init__(self, base_dataset, min_digit: int, max_digit: int, min_rotation_deg: float, max_rotation_deg: float):
        self.base_dataset = base_dataset # the original MNIST dataset
        self.min_rotation_deg = min_rotation_deg
        self.max_rotation_deg = max_rotation_deg

        # Filter dataset using max_digit:
        valid_indices = torch.where((base_dataset.targets >= min_digit) & (base_dataset.targets <= max_digit))[0]
        self.indices = valid_indices.tolist()

    def __len__(self):
        # Return the length of the filtered dataset, not the whole dataset:
        return len(self.indices)

    def __getitem__(self, idx):
        # Map the requested index to the actual index in the dataset:
        real_idx = self.indices[idx]

        # Retrieve the base image tensor and digit label:
        img_tensor, digit_label = self.base_dataset[real_idx]

        # Generate a random angle (0 to max_rotation parameter value)
        angle_deg = random.uniform(self.min_rotation_deg, self.max_rotation_deg)
        angle_rad = math.radians(angle_deg)

        # Rotate the image tensor
        img_rotated = TF.rotate(img_tensor, angle_deg, fill=0)

        img_tensor = TF.to_tensor(img_rotated)

        # Convert continuous radian angle into a 2-value tensor (Sin and Cos)
        # $$$ Not using currently, but may consider if it improves accuracy
        # sin_cos_target = torch.tensor([math.sin(angle_rad), math.cos(angle_rad)], dtype=torch.float32)

        return img_tensor, digit_label, angle_rad, angle_deg

In [4]:
train_data = RotatedMNIST(
    base_dataset=train, 
    min_digit=0, # Defaults to 0 if not found
    max_digit=9, # Defaults to 9 if not found
    min_rotation_deg=0,
    max_rotation_deg=360
)
    
test_data = RotatedMNIST(
    base_dataset=test, 
    min_digit=0,
    max_digit=9,
    min_rotation_deg=0,
    max_rotation_deg=360
)

In [19]:
#Lets VAE pick a random point int the latent space without breaking the learning.
# Turns randomness into something the network can backpropagate
def reparameterize(mu, logvar):
    # Convert log variance into standard deviation
    # std = sqrt(variance)
    # Using logvar keeps training numerically stable
    std = torch.exp(0.5 * logvar)
    
    # Sample random noise from a standard normal distribution
    # This noise is what makes the VAE stochastic
    eps = torch.randn_like(std)
    
    # Create the latent vector z
    # z = mean + (random noise * standard deviation)
    # This lets gradients flow through mu and std during training
    return mu + eps * std

In [24]:
class CerberusVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(nn.Flatten(),
                                  nn.Linear(784, 256), nn.ReLU(),
                                  nn.Linear(256, 64), nn.ReLU())
        self.mu = nn.Linear(64, 32)
        self.logvar = nn.Linear(64, 32)
        self.dec = nn.Sequential(nn.Linear(32, 64), nn.ReLU(),
                                    nn.Linear(64, 256), nn.ReLU(),
                                    nn.Linear(256, 784))
        
        self.digit_head = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 10))
        self.angle_head = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 24))

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = reparameterize(mu, logvar) if self.training else mu
        recon = self.dec(z)
        return recon, mu, logvar, self.digit_head(z), self.angle_head(z)

In [25]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    
    # Reconstruction loss:
    # Measures how close the output is to the original input
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    
    # KL divergence loss:
    # Penalizes the latent distribution for drifting too far
    # from the standard normal distribution
    kl_loss = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )
    
    # Total loss balances reconstruction quality
    # and latent space regularization
    return recon_loss + beta * kl_loss

In [ ]:
net = CerberusVAE()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

for epoch in range(8):
    net.train()
    for images, digit_labels, _, angles in train_loader:
        angle_labels = (angles // 15).long().clamp(max=23)
        opt.zero_grad()
        recon, mu, logvar, digit_output, angle_output = net.forward(images)
        loss = F.cross_entropy(digit_output, digit_labels) + F.cross_entropy(angle_output, angle_labels) + vae_loss(recon, images.flatten(1), mu, logvar)
        loss.backward()
        opt.step()

net.eval()
all_X, all_y, all_angles = [], [], []
with torch.no_grad():
    for images, labels, _, angles in test_loader:
        all_X.append(images)
        all_y.append(labels)
        all_angles.append(angles)

X = torch.cat(all_X, dim=0)
y = torch.cat(all_y, dim=0).numpy()
angle = torch.cat(all_angles, dim=0).numpy()

angle_class = (angle // 15).astype(int).clip(max=23)
angle_bin   = (angle // 30).astype(int)

with torch.no_grad():
    recon, mu, logvar, digit_out, angle_out = net(X)
    digit_pred = digit_out.argmax(1).numpy()
    angle_pred = angle_out.argmax(1).numpy()

    print('Test Digit Accuracy:', (digit_pred == y).mean())
    print('Test Rotation Accuracy:', (angle_pred == angle_class).mean())

    diff = np.abs(angle_pred - angle_class)
    diff = np.minimum(diff, 24 - diff)
    print('Rotation within 15 Degrees:', (diff <= 1).mean())

    # Reconstruction quality: error
    x_flat = X.flatten(1)                  # (N, 784)
    recon_img = torch.sigmoid(recon)       # logits -> pixel values in [0, 1]
    recon_bce = F.binary_cross_entropy_with_logits(recon, x_flat, reduction='none').sum(1)  # per image
    recon_mse = ((recon_img - x_flat) ** 2).mean(1)                                          # per image
    print('Recon BCE (per image):     ', recon_bce.mean().item())
    print('Recon MSE (per pixel):     ', recon_mse.mean().item())

    # Representations for RSA
    h1 = net.body[:3](X).numpy() # Flatten -> Linear(256) -> ReLU
    h2 = net.body[:5](X).numpy() # ... -> Linear(64) -> ReLU
    z  = mu.numpy()              # latent means

layers = {'Layer 1 (256 units)': h1,
          'Layer 2 (64 units)': h2,
          'Latent mu (32)': z}

ValueError: too many values to unpack (expected 2)

In [ ]:
labels = ['Digit', 'Angle (exact)', 'Angle (±15°)']
baseline    = [0.9048, 0.71, 0.9383]
constrained = [0.9002, 0.5693, 0.9351]

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w/2, baseline, w, label='Baseline')
b2 = ax.bar(x + w/2, constrained, w, label='Constrained')
ax.bar_label(b1, fmt='%.2f')
ax.bar_label(b2, fmt='%.2f')

ax.set_xticks(x, labels)
ax.set_ylabel('Accuracy')
ax.set_title('Baseline vs Constrained')
ax.legend()
plt.show()


In [ ]:
# # Turns input data into compact latent representation (mu and log var)
# class Encoder(nn.Module):        # Define the encoder as a neural network module
#     def __init__(self, input_dim, latent_dim):
#         super().__init__()        # Initialize the parent nn.Module class
        
#         # First fully connected layer:
#         # Takes the input data and maps it to a hidden representation
#         self.fc1 = nn.Linear(input_dim, 128)
        
#         # Linear layer that outputs the mean (mu) of the latent distribution
#         self.fc_mu = nn.Linear(128, latent_dim)
        
#         # Linear layer that outputs the log variance (log σ²) of the latent distribution
#         # We use log variance for numerical stability
#         self.fc_logvar = nn.Linear(128, latent_dim)

#     def forward(self, x):
#         # Pass input through the first layer and apply ReLU activation
#         # This extracts useful features from the raw input
#         h = F.relu(self.fc1(x))
        
#         # Compute the mean of the latent distribution
#         mu = self.fc_mu(h)
        
#         # Compute the log variance of the latent distribution
#         logvar = self.fc_logvar(h)
        
#         # Return both parameters so we can sample from the distribution later
#         return mu, logvar

# #Decoder reconstructs the original data from a sampled latent vector, trying to match the input.
# class Decoder(nn.Module):

#     def __init__(self, latent_dim, output_dim):
#         # Initialize the parent PyTorch Module
#         super().__init__()
        
#         # First fully connected layer
#         # Takes the latent vector z and expands it to a hidden representation
#         self.fc1 = nn.Linear(latent_dim, 128)
        
#         # Output layer
#         # Maps the hidden representation back to the original input size
#         # This produces the reconstruction
#         self.fc_out = nn.Linear(128, output_dim)

#     def forward(self, z):
#         # Pass the latent vector z through the first linear layer
#         # and apply ReLU to introduce non-linearity
#         h = F.relu(self.fc1(z))
        
#         # Pass the hidden representation through the output layer
#         # to produce the reconstructed input
#         return self.fc_out(h)

# class VAE(nn.Module):

#     def __init__(self, input_dim, latent_dim):
#         super().__init__()
        
#         # Encoder maps input data -> latent distribution (mu, logvar)
#         self.encoder = Encoder(input_dim, latent_dim)
        
#         # Decoder maps latent vector z -> reconstructed input
#         self.decoder = Decoder(latent_dim, input_dim)

#     def forward(self, x):
#         # Pass input through encoder to get latent distribution parameters
#         mu, logvar = self.encoder(x)
        
#         # Sample a latent vector z using the reparameterization trick
#         z = reparameterize(mu, logvar)
        
#         # Decode the latent vector back to input space
#         recon_x = self.decoder(z)
        
#         # Return reconstruction and latent stats (needed for loss)
#         return recon_x, mu, logvar

def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    
    # Reconstruction loss:
    # Measures how close the output is to the original input
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    
    # KL divergence loss:
    # Penalizes the latent distribution for drifting too far
    # from the standard normal distribution
    kl_loss = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )
    
    # Total loss balances reconstruction quality
    # and latent space regularization
    return recon_loss + beta * kl_loss


In [ ]:
# Create an optimizer that will update the VAE's parameters
# Adam is a common, stable choice for training neural networks
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

# Loop over the dataset multiple times
for epoch in range(num_epochs):
    
    # Loop over batches of data
    for batch in dataloader:
        
        # Get the input data for this batch
        x = batch
        
        # Forward pass:
        # Encode -> sample latent z -> decode
        recon_x, mu, logvar = vae(x)
        
        # Compute the VAE loss (reconstruction + KL divergence)
        loss = vae_loss(recon_x, x, mu, logvar)
        
        # Clear old gradients from the previous step
        optimizer.zero_grad()
        
        # Backpropagation:
        # Compute gradients of the loss w.r.t. model parameters
        loss.backward()
        
        # Update model parameters using the optimizer
        optimizer.step()